In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t090'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "Product Catalog Traffic Drop Alert" and occurred between 2026-04-12 21:26:08 and 2026-04-12 21:36:32 (UTC+8). The entity that triggered the alert is the operation `ListProducts` of the `ProductCatalogService` in the APM domain.

The original code execution output of IPython Kernel is also provided below for reference:

('product-catalog 流量下跌告警', {'start': '2026-04-12T21:26:08+08:00', 'end': '2026-04-12T21:36:32+08:00'}, {'entity_id': '455f921f0afc16175724d9f27faf18e4', 'entity_name': 'product-catalog::oteldemo.ProductCatalogService/ListProducts', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file to identify all available KPIs
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# Extract unique metric names to identify available KPIs
available_kpis = metrics_data['metric'].unique()
available_kpis

```
Out[2]:
```
The available KPIs include metrics related to node status, resource usage (CPU, memory, disk), deployment replicas and availability, cluster and namespace pod counts, API server requests, JVM performance (GC, memory, threads), thread pool metrics, and application-level metrics such as `request_count`, `error_count`, `error_rate`, `latency`, and `slow`. These KPIs are relevant for analyzing the traffic drop issue.

The original code execution output of IPython Kernel is also provided below for reference:

['node_ready_status' 'node_not_ready_status' 'node_memory_usage_rate'
 'node_disk_usage_rate' 'node_pod_running_count' 'node_cpu_usage_rate'
 'deployment_desired_replicas' 'deployment_available_replicas'
 'deployment_ready_replicas' 'deployment_availability_rate'
 'deployment_memory_usage_total' 'deployment_cpu_usage_vs_requests'
 'deployment_cpu_usage_total' 'deployment_cpu_usage_vs_limits'
 'cluster_failed_node_count' 'cluster_node_count'
 'apiserver_concurrent_connections' 'cluster_number_of_not_ready_pods'
 'cluster_number_of_pod' 'apiserver_non_2xx_read_request_qps'
 'apiserver_non_2xx_write_request_qps' 'apiserver_requests_per_second'
 'namespace_number_of_not_ready_pods' 'namespace_number_of_pods'
 'slow_count' 'error_count' 'request_count' 'avg_request_latency_seconds'
 'arms_jvm_gc_total' 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'slow' 'error' 'error_rate']```
```

In [ ]:
# In[3]:

# Filter metrics data for the specific operation 'ListProducts' of 'ProductCatalogService'
filtered_metrics = metrics_data[
    metrics_data['entity_name'] == 'product-catalog::oteldemo.ProductCatalogService/ListProducts'
]

# Convert 'value' column to numeric for calculations
filtered_metrics['value'] = pd.to_numeric(filtered_metrics['value'], errors='coerce')

# Relevant KPIs for analysis
relevant_kpis = ['request_count', 'error_count', 'error_rate', 'latency', 'slow']

# Filter data for relevant KPIs
filtered_kpis = filtered_metrics[filtered_metrics['metric'].isin(relevant_kpis)]

# Calculate global thresholds (P95) for each KPI
global_thresholds = filtered_kpis.groupby('metric')['value'].quantile(0.95).reset_index()
global_thresholds.columns = ['metric', 'p95_threshold']

global_thresholds

```
Out[3]:
```
For the `ListProducts` operation of the `ProductCatalogService`, the only relevant KPI with sufficient data is `request_count`, and its global P95 threshold is 8587.6. Other KPIs (`error_count`, `error_rate`, `latency`, `slow`) do not have sufficient data for threshold calculation.

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0  request_count         8587.6```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Define the alert time window in UTC+8
alert_start = datetime.fromisoformat(alert_window['start']).astimezone(pytz.timezone('Asia/Shanghai'))
alert_end = datetime.fromisoformat(alert_window['end']).astimezone(pytz.timezone('Asia/Shanghai'))

# Filter the `request_count` KPI data for the specified time window
request_count_data = filtered_kpis[
    (filtered_kpis['metric'] == 'request_count') &
    (filtered_kpis['time'] >= alert_start.timestamp() * 1e6) &
    (filtered_kpis['time'] <= alert_end.timestamp() * 1e6)
]

# Identify anomalies where `request_count` is below the global P95 threshold
p95_threshold = global_thresholds.loc[global_thresholds['metric'] == 'request_count', 'p95_threshold'].values[0]
anomalies = request_count_data[request_count_data['value'] < p95_threshold]

anomalies

```
Out[4]:
```
During the alert time window, the `request_count` for the `ListProducts` operation of the `ProductCatalogService` consistently fell below the global P95 threshold of 8587.6. The values ranged from 2116.0 to 7136.0, indicating a significant drop in traffic for this operation. This confirms the anomaly in traffic volume during the specified period.

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                                        entity_name         metric   value                            metric_set_id service
61142  1776000368000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  6327.0  apm@metric_set@apm.metric.apm.operation    None
61143  1776000398000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5998.0  apm@metric_set@apm.metric.apm.operation    None
61144  1776000428000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5249.0  apm@metric_set@apm.metric.apm.operation    None
61145  1776000458000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5019.0  apm@metric_set@apm.metric.apm.operation    None
61146  1776000488000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5730.0  apm@metric_set@apm.metric.apm.operation    None
61147  1776000518000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  6669.0  apm@metric_set@apm.metric.apm.operation    None
61148  1776000548000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  7136.0  apm@metric_set@apm.metric.apm.operation    None
61149  1776000578000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  6324.0  apm@metric_set@apm.metric.apm.operation    None
61150  1776000608000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5413.0  apm@metric_set@apm.metric.apm.operation    None
61151  1776000638000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5874.0  apm@metric_set@apm.metric.apm.operation    None
61152  1776000668000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5153.0  apm@metric_set@apm.metric.apm.operation    None
61153  1776000698000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  2247.0  apm@metric_set@apm.metric.apm.operation    None
61154  1776000728000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  2116.0  apm@metric_set@apm.metric.apm.operation    None
61155  1776000758000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5243.0  apm@metric_set@apm.metric.apm.operation    None
61156  1776000788000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5685.0  apm@metric_set@apm.metric.apm.operation    None
61157  1776000818000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5271.0  apm@metric_set@apm.metric.apm.operation    None
61158  1776000848000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5838.0  apm@metric_set@apm.metric.apm.operation    None
61159  1776000878000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  6002.0  apm@metric_set@apm.metric.apm.operation    None
61160  1776000908000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5747.0  apm@metric_set@apm.metric.apm.operation    None
61161  1776000938000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5622.0  apm@metric_set@apm.metric.apm.operation    None
61162  1776000968000000    apm  apm.operation  455f921f0afc16175724d9f27faf18e4  product-catalog::oteldemo.ProductCatalogServic...  request_count  5664.0  apm@metric_set@apm.metric.apm.operation    None```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Convert relevant columns to numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces for the `ListProducts` operation of the `ProductCatalogService` during the alert time window
filtered_traces = traces_data[
    (traces_data['spanName'] == 'ListProducts') &
    (traces_data['serviceName'] == 'product-catalog') &
    (traces_data['startTime'] / 1e6 >= alert_start.timestamp() * 1e3) &
    (traces_data['endTime'] / 1e6 <= alert_end.timestamp() * 1e3)
]

# Analyze upstream and downstream services by extracting unique service names from the traces
upstream_downstream_services = filtered_traces['serviceName'].unique()

filtered_traces, upstream_downstream_services

```
Out[5]:
```
No trace data was found for the `ListProducts` operation of the `ProductCatalogService` during the alert time window. This indicates that either no traces were generated for this operation during the specified period or the trace data is missing, making it impossible to analyze upstream or downstream services contributing to the traffic drop.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, events, __topic__, __tag__:__pack_id__, ip, links]
Index: []

[0 rows x 21 columns], array([], dtype=object))```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Convert the `_time_` column to numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the `ListProducts` operation of the `ProductCatalogService` during the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= alert_start.timestamp() * 1e3) &
    (logs_data['_time_'] <= alert_end.timestamp() * 1e3) &
    (logs_data['content'].str.contains('ListProducts', case=False, na=False))
]

# Look for error messages or anomalies in the log content
error_logs = filtered_logs[filtered_logs['content'].str.contains('error', case=False, na=False)]

filtered_logs, error_logs

```
Out[6]:
```
No logs were found for the `ListProducts` operation of the `ProductCatalogService` during the alert time window. Consequently, no error messages or anomalies in the log content could be identified as potential causes for the traffic drop.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns], Empty DataFrame
Columns: [content, _time_, _source_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, _container_ip_, __topic__, __tag__:__pack_id__, __tag__:_node_ip_, __tag__:_cluster_id_, __tag__:__hostname__, __tag__:_node_name_]
Index: []

[0 rows x 15 columns])```
```

In [ ]:
# In[7]:

# Load the topology.json file to analyze the service dependency graph
topology_file_path = f'{task_directory}/topology.json'

with open(topology_file_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = topology_data.get('entities', [])
edges = topology_data.get('edges', [])

# Identify the entity ID for the `ListProducts` operation of the `ProductCatalogService`
list_products_entity_id = alert_entity['entity_id']

# Find upstream services or components that interact with the `ListProducts` operation
upstream_edges = [edge for edge in edges if edge['dst'] == list_products_entity_id and edge['relation'] == 'calls']
upstream_services = [entity for entity in entities if entity['id'] in {edge['src'] for edge in upstream_edges}]

upstream_edges, upstream_services

```
Out[7]:
```
The service dependency graph does not show any upstream services or components interacting with the `ListProducts` operation of the `ProductCatalogService`. This indicates that no direct dependencies or callers were identified for this operation in the topology.

The original code execution output of IPython Kernel is also provided below for reference:

([], [])```
```